# Conduite — V-JEPA fidèle + System-2 (planner MPC latent)

Pipeline **fidèle à la vision LeCun** (`vjepa.py` : encodeur-contexte sur tokens visibles, prédicteur **attentionnel**, masquage par tubelets, multi-masque, SIGReg sans EMA/stop-grad).

1. **Transfert** UCF101 → dashcam Nexar (résolution H=96)
2. **Rollout collision** : on masque le futur, le World Model l'imagine, risque + heatmap
3. **System-2 (MPC latent)** : le World Model imagine les futurs sous chaque action et choisit la meilleure décision

Repo : https://github.com/frpatry/jepa-physics-bench

## 1. Récupérer le code + installer lejepa (`git pull` = dernière version)

In [ ]:
import os
if not os.path.exists('/content/jepa-physics-bench'):
    !git clone https://github.com/frpatry/jepa-physics-bench.git /content/jepa-physics-bench
!cd /content/jepa-physics-bench && git pull
%cd /content/jepa-physics-bench
!pip install -q lejepa || pip install -q git+https://github.com/rbalestr-lab/lejepa
print('OK — code à jour, lejepa installé (cv2 + huggingface_hub déjà sur Colab)')

## 2. Transfert UCF101 → dashcam Nexar (H=96)
Encodeur V-JEPA entraîné sur UCF101 (générique), **gelé**, appliqué aux dashcams pour détecter le danger. `#2 GRAAL` = transfert cross-domaine vs `#1` in-domaine vs hasard 0.50.

Si **OOM** : baisser `--bs` (ex. 8) ou `--H 64`.

In [ ]:
!python -u driving_transfer.py --n_nexar 400 --ucf_source full --ucf_nclass 50 --ucf_per 40 --H 96 --bs 16

## 3. Prédiction de collision par rollout latent (+ heatmap) — **par transfert**
World model V-JEPA pré-entraîné sur **vidéo variée UCF101** (`--ssl_source ucf`), **gelé**, puis on transfère seulement la tête collision sur la dashcam. On masque les frames futures, le prédicteur attentionnel les **imagine**, K futurs échantillonnés → risque + GIF (image + heatmap + jauge).

⚠️ `--ucf_source full` télécharge UCF101 complet (~6 Go) au 1er run. Pour aller plus vite : `--ucf_source subset` (10 classes).

In [ ]:
!python -u driving_rollout.py --ssl_source ucf --ucf_source full --ucf_nclass 40 --ucf_per 30 --n_nexar 800 --H 96 --patch 8 --T 16 --k_viz 3 --bs 8

In [ ]:
# afficher les GIF produits par le rollout
from IPython.display import Image, display
import glob
for p in sorted(glob.glob('/content/rollout_clip*.gif')): display(Image(p))

In [ ]:
# COMPARAISON : world model in-domaine (dashcam seule) vs varié (UCF101).
# La thèse : le world model VARIÉ anticipe-t-il mieux la collision (risque collisions >> normales) ?
!python -u driving_rollout.py --ssl_source nexar --n_nexar 800 --H 96 --patch 8 --T 16 --k_viz 0 --bs 8

## 4. System-2 — planner MPC latent (boucle fermée)
World model V-JEPA gelé → dynamique action-conditionnée `g(z,a)→z'` + tête danger `c(z)`. À chaque pas le planner imagine les futurs sous chaque frein candidat et choisit l'argmin (danger + coût).

Compare en boucle fermée : **naïf → réactif → MPC**. Surveiller la ligne `c(z) moyen : danger=… sûr=…` (doit bien séparer) et le `pred` de la dynamique (si bloqué à 0.000 → pousser `--steps` pour enrichir l'encodeur).

Compromis sécurité/confort réglable : `--w_coll` `--w_prog` `--horizon`.

In [ ]:
!python -u drive_plan.py --steps 2500 --dyn_steps 2000 --n_train 600 --n_test 150